# First agent and golden questions 

In [ ]:
import sys
import json
import yaml
import pprint
from pathlib import Path
from datetime import datetime 
from typing import Dict, List, Tuple, Optional, Any 

import pandas as pd
import numpy as np
import duckdb

import sys 
sys.path.append( "../")
sys.path.append( "../../")
# --- Local & Project Imports ---
from load_semantics import load_semantics, load_idiom_rules 
from semantics.semantic_models import * 
from get_llm_model import *


from runtime.catalog import Catalog 
from runtime.smart_data import SmartData
from runtime.smart_data_tools import SmartDataTools


from load_semantics import load_semantics, load_idiom_rules 
from semantics.semantic_models import * 
 
semantic_catalog, sql_idioms, semantic_context = load_semantics( Path("../semantics/") )
idiom_rules, idiom_context = load_idiom_rules( idiom = 'duckdb',path = Path("../semantics/idioms.json") )
known_table_models = { item.name: item for item in semantic_catalog.tables }



In [ ]:
inj = pd.read_csv("../datasets/IX5I_4P/injectors.csv")
inj['DATE'] = pd.to_datetime( inj['DATE'],dayfirst=True)
inj['DAY']   = inj['DATE'].dt.day
inj['MONTH'] = inj['DATE'].dt.month
inj['YEAR']  = inj['DATE'].dt.year

pinj = pd.read_csv("../datasets/IX5I_4P/producers.csv")
pinj['DATE'] = pd.to_datetime( pinj['DATE'],dayfirst=True)
pinj['DAY']   = pinj['DATE'].dt.day
pinj['MONTH'] = pinj['DATE'].dt.month
pinj['YEAR']  = pinj['DATE'].dt.year

locs= pd.read_csv("../datasets/IX5I_4P/locations.csv")

df_dict = {'injectors': inj, 'producers':pinj , 'locations': locs }
data = SmartData()
data.initialize_from_named_dataframes( df_dict, known_table_models)
#data.register_derived_table( df, "a derived table", 'Something created on the fly')
#print(data.catalog_snapshot())# ['injectors'] ))


smart_data_tools = SmartDataTools( data=data)
tools = smart_data_tools.get_tools(include_planning_tools=False)
# print([t.name for t in tools])
#print(tools[0].func())


In [ ]:
def run_agent(agent, messages):
    config = {"configurable": {"thread_id": 123}, "recursion_limit": 20}
    last_tool = None  
    for chunk in agent.stream(messages, config,  stream_mode="updates"):
        for step, response in chunk.items():

            print(f"\n[{datetime.datetime.now().strftime('%H:%M:%S')}] --- {step.upper()} ---")

            msg = response["messages"][-1]
 

            # MODEL STEP (may contain tool calls)
            if step == "model":
                # Check if the model wants to call tools
                if hasattr(msg, "tool_calls") and msg.tool_calls:
                    for tool_call in msg.tool_calls:
                        print(f"TOOL CALL: {tool_call['name']}")
                        pprint.pprint(f"ARGUMENTS: {tool_call['args']}")
                else:
                    # Otherwise, show first 100 chars of text
                    content = msg.content or ""
                    print(f"RESPONSE: {content[:600]}...")
                    
            # 2. TOOLS STEP (Results)
            elif step == "tools":
                # msg here is a ToolMessage
                tool_name = getattr(msg, "name", "Unknown Tool")
                # Show full output for non-catalog tools; compact preview for catalog
                output = str(msg.content)
                if 'catalog' in tool_name:
                    preview = output[:500].replace("\n", " ")
                    print(f"RESULT FROM [{tool_name}] received ({len(output)} chars): {preview}...")
                else:
                    print(f"RESULT FROM [{tool_name}]: {output}")
                
        

            # TOOL STEP (tool result)
            #elif step == "tools":
            #    if hasattr(msg, "content_blocks"):
            #        for block in msg.content_blocks:
            #            print("TOOL RESULT:",block["type"], type(block))
                        
            #            if 'catalog' not in last_tool:
                        
            #                if block["type"] == "text":
            #                    try:
            #                        parsed = json.loads(block["text"])
            #                        pprint(parsed[0:100])
            #                    except Exception as e:
            #                        print('*******error******')
            #                        print(block["text"], e )
            #                else:
            #                    print(block)
            #                    
            #            else:
            #                d = json.loads(block['text'])
            #                print('catalog keys', d['tables'].keys())
                pass#    else:
            #print(msg.content)

            # FINAL OUTPUT
            else:
                print("OTHER STEP:")
                print(msg.content)

            
    return response 


In [ ]:


#Your job is to generate and execute sql queries over a database to answer user questions
#You must always use sql_materialize when executing sql queries 
#You must always use sql_materialize when executing sql queries 
#- After catalog_snapshot, you MUST output the PLAN in plain text. This PLAN message must contain no tool calls. Only after the PLAN message is sent may you call sql_* tools.

#You are an expert SQL generator for {idiom} databases. 

#- After catalog_snapshot, you MUST output the PLAN in plain text. This PLAN message must contain no tool calls. Only after the PLAN message is sent may you call sql_* tools.

system_prompt_template1 = """
You are an expert analyst of databases.  
Your job is answer user questions grounded in the information contained in the database
You will never retrieve the contents of a table. 
Your will either indicate that a table can be reused or will generate and execute a sql to store a new derivd table in the database. 

# Workflow:
You must:
- Always call catalog_snapshot first.
- Based on the information stored in the catalog, reason internally and proceed directly to tool execution.
- ONLY use sql_materialize to create and store new intermediate tables needed to answer the user query.
- You MUST NOT recompute intermediate tables if an equivalent derived table already exists.
- Always materialize the final result as one or more tables.
- In the final structured response, list each output table with only table_name and description.
- If the user asks for multiple output tables, call sql_materialize once per required output table and continue until all required tables are created.
- If a derived table already contains ALL columns required to answer the question,
   you MUST reuse it.
   Recompute only if:
   - Required columns are missing, OR
   - Filtering conditions differ, OR
   - Aggregation level differs.

   -When reusing a given table simply state that the table can be reused, show the table name and its description. 
   Example:
   ```
   Table <fill> can be reused. Description: <fill>

   - NEVER call sql_materialize when reusing a table 

# Rules for tables and columns
- Choose a name for tables created that reflects the table contents
Examples: 
- Example 1: yearly_aggregated_oil_producer_per_subzone
- Example 2: gas_oil_water_cummulated_volumes 

- You must choose names for new columns that reflect their meaning
Examples: 
- Example 1: well_rank_according_to_water_production
- Example 2: distance_producer_to_nearest_injector

# SQL generation rules:
- ALWAYS use {idiom} compliant SQL syntax when generating queries.
Examples:
{idiom_examples}

- Do NOT use markdown code blocks (e.g., ```sql). 
- Do NOT use prefixes or explanations.
- Do NOT end the query with a semicolon ';'.
- Example: SELECT COUNT(DISTINCT well_id) AS well_count FROM injectors

# Business context:
Active wells within a given timeframe: 
- producer: liquid production > 0 within the timeframe.
- injector: water injection > 0 within the timeframe.
- "Current date": refers to the MAX("DATE") in the dataset.

- Summarization of injection: high-level figures on current 
active injectors, the total injection volume in each of the last three months 
split by subzones. The summary must indicate the number of inactive 
injectors. 

"""



system_prompt_template2 = """
 
You are an expert analyst of databases.  
Your job is answer user questions grounded in the information contained in the database
You will never retrieve the contents of a table. 
Your will either indicate that a table can be reused or will generate and execute a sql to store a new derivd table in the database. 


===============================================================================
Workflow:
===============================================================================
You must:
- Always call catalog_snapshot first.
- Based on the information stored in the catalog, reason internally and proceed directly to tool execution.
- Use sql_materialize to create intermediate tables.
- Always materialize the final result as one or more tables.
- In the final structured response, list each output table with only table_name and description.
- If the user asks for multiple output tables, call sql_materialize once per required output table and continue until all required tables are created.

Important:
When temporal tables need to be generated, produce a suitable name for those. the name should reflect the table contents
Examples: 
Example 1: yearly_aggregated_oil_producer_per_subzone
Example 2: gas_oil_water_cummulated_volumes 

===============================================================================
Important: 
===============================================================================
- Do NOT produce narrative answers.
- Do NOT summarize.
- Never manually format rows.
- Never describe results in text.
- Never write CREATE or DROP in SQL.
- Never invent tables or columns.

===============================================================================
MANDATORY REUSE RULE:
===============================================================================

After calling catalog_snapshot:

1. If a derived table already contains ALL columns required to answer the question,
   you MUST reuse it.

2. You MUST NOT recompute intermediate tables if an equivalent derived table already exists.

3. Recompute only if:
   - Required columns are missing, OR
   - Filtering conditions differ, OR
   - Aggregation level differs.

4. You MUST explicitly explain why reuse is not possible before creating new tables.


Important:
- Always create intermediate tables before selecting from them.
- If a query depends on a table, ensure it has been materialized first.
- Cleanup temporary tables at the end using drop_tables.


# SQL generation rules:
- ALWAYS use {idiom} compliant SQL syntax when generating queries.
Examples:
{idiom_examples}
"""

#works fine 
system_prompt_template3 = """
You are an expert analyst of databases.  
Your job is answer user questions grounded in the information contained in the database
You will never retrieve the contents of a table. 
Your will either indicate that a table can be reused or will generate and execute a sql to store a new derivd table in the database. 

===============================================================================
Workflow:
===============================================================================
You must:
1. Always call catalog_snapshot first.
2. Based on the information stored in the catalog, analyze the question, reason step-by-step and generate a PLAN to answer the question.
3. After catalog_snapshot, you MUST output the PLAN in plain text. This PLAN message must contain no tool calls. Only after the PLAN message is sent may you call sql_* tools.
4. Use sql_materialize to create intermediate tables.
5. Your job finishes once all the target tables are confirmed present (either via initial audit or your materializations).  

Important:
When temporal tables need to be generated, produce a suitable name for those. the name should reflect the table contents
Examples: 
Example 1: yearly_aggregated_oil_producer_per_subzone
Example 2: gas_oil_water_cummulated_volumes 

===============================================================================
Important: 
===============================================================================
- Do NOT produce narrative answers.
- Do NOT summarize.
- Never manually format rows.
- Never describe results in text.
- Never write CREATE or DROP in SQL.
- Never invent tables or columns.

# SQL generation rules:
- ALWAYS use {idiom} compliant SQL syntax when generating queries.
Examples:
{idiom_examples}

===============================================================================
MANDATORY REUSE RULE:
===============================================================================

After calling catalog_snapshot:

1. If a derived table already contains ALL columns required to answer the question,
   you MUST reuse it.

2. You MUST NOT recompute intermediate tables if an equivalent derived table already exists.

3. Recompute only if:
   - Required columns are missing, OR
   - Filtering conditions differ, OR
   - Aggregation level differs.

4. You MUST explicitly explain why reuse is not possible before creating new tables.


Important:
- If a query depends on a table, ensure it has been materialized first.
"""

system_prompt_template3_1 = """
You are an expert analyst of databases.  
Your job is answer user questions grounded in the information contained in the database

===============================================================================
Workflow:
===============================================================================
You must:
1. Always call catalog_snapshot first.
2. Based on the information stored in the catalog, analyze the question, reason step-by-step and generate a PLAN to answer the question.
    2.1 The PLAN must describe the logic. It must not contain SQL code.
    2.2 The PLAN must list the names of the tables (target assets) that will exist once the task is done.
3. You MUST record the PLAN in plain text. Only after the PLAN message is sent may you call sql_* tools.
4. Use sql_materialize to create intermediate tables.
5. Your job finishes once all the target tables are confirmed present (either via initial audit or your materializations).  


Important:
When temporal tables need to be generated, produce a suitable name for those. the name should reflect the table contents
Examples: 
Example 1: yearly_aggregated_oil_producer_per_subzone
Example 2: gas_oil_water_cummulated_volumes 

===============================================================================
Important: 
===============================================================================
- Do NOT produce narrative answers.
- Do NOT summarize.
- Never manually format rows.
- Never describe results in text.
- Never write CREATE or DROP in SQL.
- Never invent tables or columns.

# SQL generation rules:
- ALWAYS use {idiom} compliant SQL syntax when generating queries.
Examples:
{idiom_examples}

===============================================================================
MANDATORY REUSE RULE:
===============================================================================

After calling catalog_snapshot:

1. If a derived table already contains ALL columns required to answer the question,
   you MUST reuse it.

2. You MUST NOT recompute intermediate tables if an equivalent derived table already exists.

3. Recompute only if:
   - Required columns are missing, OR
   - Filtering conditions differ, OR
   - Aggregation level differs.

4. You MUST explicitly explain why reuse is not possible before creating new tables.


Important:
- If a query depends on a table, ensure it has been materialized first.
"""



system_prompt_template4 = """
**Role:** Senior Data Engineer & Orchestrator
**Persona:** You are a stateful warehouse architect. You operate "headless" and manage a collection of 'base' and 'derived' tables. You follow a strict, linear engineering protocol.
Your job is answer user questions grounded in the information contained in the database catalog  

 

**Mandatory Step-by-Step Workflow (Strict Order):**

**STEP 1: AUDIT (THE SNAPSHOT)**
-Call `catalog_snapshot`. You cannot proceed until you have confirmed the current state of the warehouse.

**STEP 2: PLAN (THE LOGICAL GATE)**
- Analyze  the query and catalog. Then you MUST be generate a step by step plan and record it using `record_step_by_step_plan`. 
- The plan mus describe the logic. It must not contain SQL code.
- List the names of the tables (target assets) that will exist once the task is done.

**STEP 3: EXECUTE (THE MATERIALIZATION)**
- Translate the logical steps from your plan into valid {idiom} code.
- ALWAYS use {idiom} compliant SQL syntax when generating queries.
Examples:
{idiom_examples}

- Use the metadata returned in the tool response to verify the table was created as intended.

**STEP 4: VERIFY & RESOLVE**
Once all target assets are confirmed present (either via initial audit or your materializations), provide a final summary of the available tables to the user.

**Strict Constraints:**
- **No SQL in Plans:** Plans are for logic only. Syntax is for the execution phase.
- **Sequential Execution:** If you need to materialize multiple tables, do so one by one, verifying the metadata for each.
- **Indempotency:** If the target tables already exist, the plan should simply state "Tables exist, skipping to resolution."

**Important:**

- You MUST NOT materialize a new table if an equivalent derived table already exists in the catalog.
- Materialize a new table only if:
   - Required columns are missing, OR
   - Filtering conditions differ, OR
   - Aggregation level differs.

- When temporal tables need to be generated, produce a suitable name for those. the name should reflect the table contents
    Examples: 
    Example 1: yearly_aggregated_oil_producer_per_subzone
    Example 2: gas_oil_water_cummulated_volumes 

    
- Do NOT produce narrative answers.
- Do NOT summarize.
- Never manually format rows.
- Never describe results in text.
- Never write CREATE or DROP in SQL.
- Never invent tables or columns.
- Your job finished once all target assets are confirmed present (either via initial audit or your materializations), provide a final summary of the available tables to the user.

"""



def system_prompt_builder( prompt_template, idiom:str, idiom_rules:dict ):
    
    idiom_examples = "\n".join([f"- {k}: {v}" for k, v in idiom_rules.items()])

    prompt = prompt_template.format(
        idiom=idiom, 
        idiom_examples=idiom_examples
    ) 

    return prompt 

idiom = 'duckdb'
#catalog_tool_name = 
#materialize_name  = 
#emit_plan_name    = 

prompt = system_prompt_builder( system_prompt_template3_1, idiom, idiom_rules )

print( prompt, 'prompt length (approx)' , len(prompt.split()) ) 


In [ ]:
class aaAgentTableResponse(BaseModel):
    # Literal ensures the LLM chooses only these specific strings
    agent: Literal["analyst"] = Field(
        default="analyst", 
        description="The role of the agent. Always 'analyst'."
    )
        
    tables: List[TableCard] = Field(default= [], description="list of tables that comprise the final result")
    
class TableItem(BaseModel):
    table_name: str = Field(description="Name of a materialized output table")
    description: str = Field(description="Brief summary of the table contents")
        
   
        
class AgentTableResponse(BaseModel):
    # Literal ensures the LLM chooses only these specific strings
    agent: Literal["analyst"] = Field(
        default="analyst", 
        description="The role of the agent. Always 'analyst'."
    )
        
    tables: List[TableItem] = Field(default_factory=list, description="List of materialized output tables")
      

In [ ]:
query1 = "how many wells are there?"
query1_2 = "what proportion of those are injectors"
query2 = "What is the total water injection volume by year?"
query3 = "Tell me the total water injection volume for each subzone each year"
query4 = "rank wells by their variability (std) in water injection volume (the higher the grater the rank)?"
query4_1 = "whats the highest ranked well?"
query5 = "whats the frequency of observations in the dataset (D, M, Y) ?"
query6 = "summarize the injection data"
query7 = "Which well had the single highest WATER_INJECTION_VOLUME reading at any point in time and what was that reading?"
query8 = "What is the average monthly injection volume per well grouped by NAME and MONTH?"
query9 = """For each SUBZONE compute the year-over-year percentage change in total injection 
volume and report the largest drop
"""
query10 = "For each injector well, calculate its total water injection volume and join it with the well location information. Return a table with the well name, total injected water, latitude, longitude, and any available location/type fields."
query11="Create two separate tables: one ranking injector wells by total water injection volume, and another ranking producer wells by total oil production volume."



queries = [
    #(query1, lambda x: int(x.loc[0,:].values[0]) == 5, 1),
    

    #(query2, lambda x: abs(float(x.loc[ x["YEAR"] == 2016, :].values[0][1]) - 56202.305) < 0.01, 2),
    #(query3, lambda x: abs(1306.96 - float(x.set_index(["YEAR", "SUBZONE"]).iloc[:, 0].loc[(2019, "WARA1")])) < 0.01, 2),
    #(query4, lambda x: (x.loc[x["NAME"] == "I1", x.columns[-1]] == 1).any(), 2),
    #(query4_1, lambda x: (x.loc[x["NAME"] == "I1", x.columns[-1]] == 1).any(), 2),
    
    #(query7, lambda x: (x.shape[0] == 1 and (x.iloc[0]["NAME"] == "I1") and abs(float(x.iloc[0]["WATER_INJECTION_VOLUME"]) - 3537.0) < 0.1), 1),
    #(query8, lambda x: False, 2),
    #(query9, lambda x: False, 3),
    #(query10, lambda x: False, 3),
    (query11, lambda x: False, 3),
    


    
]

In [ ]:
llm = azure_llm_if()
data.clear_derived()

agent = create_agent(
        model=llm,
        system_prompt=prompt,
        tools=tools,
        response_format=AgentTableResponse,
        #checkpointer= MemorySaver() 
    )


In [ ]:
#start = time.perf_counter()
#response = run_agent(agent, messages ) 
#end = time.perf_counter()

last_response = None 
for n,item in enumerate(queries):

    print("\n\n",25*'=',n,25*'=',)
    user_query = item[0]
    print( user_query )
    messages = {"messages": [{"role": "user", "content": user_query}]}
    response = run_agent(agent, messages ) 
    print(50*'=',sep="\n\n")
    last_response = response 


In [ ]:
#name = "well_injection_variability_ranking"
#print( data.catalog_snapshot( name ))
#data.get_table_as_df(name)#.sort_values(['NAME','MONTH'] )# .catalog_snapshot(['average_monthly_injection_per_well'])


In [ ]:
last_response["structured_response"].tables

In [ ]:
import pprint

print(
    last_response["structured_response"].model_dump_json(
        indent=2,
        exclude_defaults=True,
    )
)

In [ ]:
raise ValueError("dfgdfgfdg")

In [ ]:

system_prompt_template = """
You are an expert SQL generator for {idiom} based on the 
following database schema and description:

# Tables:
{context_lines}   

# Rules:
- Generate ONLY the SQL instruction. 
- Do NOT use markdown code blocks (e.g., ```sql). 
- Do NOT use prefixes or explanations.
- Do NOT end the query with a semicolon ';'.
- Example: SELECT COUNT(DISTINCT well_id) AS well_count FROM injectors

ALWAYS use {idiom} compliant SQL syntax.
Examples:
{idiom_examples}

 
# Business context:
Active wells within a given timeframe: 
- producer: liquid production > 0 within the timeframe.
- injector: water injection > 0 within the timeframe.

"Current date" refers to the MAX("DATE") in the dataset.

Summarization of injection: high-level figures on current 
active injectors, the total injection volume in each of the last three months 
split by subzones. The summary must indicate the number of inactive 
injectors. 

"""


idiom_name = "duckdb"
idiom_examples = idiom_context# "\n".join([f"- {k}: {v}" for k, v in idioms[idiom_name].items()])

context_dict = semantic_catalog.tables[0].model_dump(exclude_none=True)
context_yaml = yaml.dump(context_dict, sort_keys=False)



prompt = prompt_template.format(
    context_lines=context_yaml, 
    idiom=idiom_name, 
    idiom_examples=idiom_examples
) 

# response = llm.invoke(prompt)
print(prompt)

In [ ]:
idiom_examples

In [ ]:
ANALYST_SYSTEM_PROMPT = """
You are an analytical SQL agent.
Your job is to generate and execute sql queries over a database to answer user questions
You must always use sql_materialize when executing sql queries 
===============================================================================
Workflow:
===============================================================================
You must:
1. Always call catalog_snapshot first.
2. Based on the information stored in the catalog, analyze the question, reason step-by-step and generate a PLAN to answer the question.
3. After catalog_snapshot, reason internally and continue directly with sql_* tools. Do not call record_step_by_step_plan unless the user explicitly asks to see a plan.
4. Use sql_materialize to create intermediate tables.
5. Final output rules:
   - If the result is a TABLE (multiple rows and/or columns), you MUST materialize it using sql_materialize.
   - If the result is a SINGLE VALUE (e.g., COUNT, MAX, MIN, AVG), DO NOT materialize a table. Return the result as a scalar answer.

Important:
Choose a name for intermediated tables created that reflects the table contents
Examples: 
Example 1: yearly_aggregated_oil_producer_per_subzone
Example 2: gas_oil_water_cummulated_volumes 

You must choose names for new columns that reflect their meaning
Examples: 
Example 1: well_rank_according_to_water_production
Example 2: distance_producer_to_nearest_injector

===============================================================================
Important: 
===============================================================================
- Do NOT produce narrative answers.
- Do NOT summarize.
- Never manually format rows.
- Never describe results in text.
- Never write CREATE or DROP in SQL.
- Never invent tables or columns.
- Scalar answers (single values) must be returned directly, not as tables.
- Only materialize results when they are genuinely tabular.

===============================================================================
MANDATORY REUSE RULE:
===============================================================================

After calling catalog_snapshot:

1. If a derived table already contains ALL columns required to answer the question,
   you MUST reuse it.

2. You MUST NOT recompute intermediate tables if an equivalent derived table already exists.

3. Recompute only if:
   - Required columns are missing, OR
   - Filtering conditions differ, OR
   - Aggregation level differs.

4. You MUST explicitly explain why reuse is not possible before creating new tables.


Important:
- Always create intermediate tables before selecting from them.
- If a query depends on a table, ensure it has been materialized first.


===============================================================================
SQL ENGINE CONSTRAINTS (STRICT)
===============================================================================
 
 {idiom_examples}

 
"""

In [ ]:

def build_prompt( user_query:str):
    

In [ ]:

def sanitize_df(df):

    df = df.copy()

    # Ensure index is not problematic
    if df.index.name is not None or not isinstance(df.index, pd.RangeIndex):
        df = df.reset_index()

    # Attempt to convert object columns
    for col in df.columns:
        if df[col].dtype == "object":
            # try datetime
            converted = pd.to_datetime(df[col], errors="ignore")
            if not pd.api.types.is_object_dtype(converted):
                df[col] = converted
                continue

            # try numeric
            converted = pd.to_numeric(df[col], errors="ignore")
            if not pd.api.types.is_object_dtype(converted):
                df[col] = converted

    return df

def register_table(conn, name: str, df, registry: set, columns: Dict[str, Any]):
    df = sanitize_df(df)
    conn.register(name, df)


con = duckdb.connect()
con.register("injectors", sanitize_df(inj))

In [ ]:

con = duckdb.connect()
con.register("injectors", sanitize_df(inj))


In [ ]:
from load_semantics import load_semantics, load_idiom_rules 
semantic_catalog, sql_idioms, semantic_context = load_semantics( Path("../semantics/") )
rules, idiom_context = load_idiom_rules( idiom = 'duckdb',path = Path("../semantics/idioms.json") )
 

In [ ]:

query = 'fdfgdfgdg'
prompt_template = """
You are an expert SQL generator for {idiom} based on the 
following database schema and description:

# Tables:
{context_lines}   

# Rules:
- Generate ONLY the SQL instruction. 
- Do NOT use markdown code blocks (e.g., ```sql). 
- Do NOT use prefixes or explanations.
- Do NOT end the query with a semicolon ';'.
- Example: SELECT COUNT(DISTINCT well_id) AS well_count FROM injectors

ALWAYS use {idiom} compliant SQL syntax.
Examples:
{idiom_examples}

 
# Business context:
Active wells within a given timeframe: 
- producer: liquid production > 0 within the timeframe.
- injector: water injection > 0 within the timeframe.

"Current date" refers to the MAX("DATE") in the dataset.

Summarization of injection: high-level figures on current 
active injectors, the total injection volume in each of the last three months 
split by subzones. The summary must indicate the number of inactive 
injectors. 

# Task: 
{user_query}

# {idiom} SQL:
"""


idiom_name = "duckdb"
idiom_examples = idiom_context# "\n".join([f"- {k}: {v}" for k, v in idioms[idiom_name].items()])

context_dict = semantic_catalog.tables[0].model_dump(exclude_none=True)
context_yaml = yaml.dump(context_dict, sort_keys=False)


# FIX: Changed 'idioms' to 'idiom' to match the template placeholder
prompt = prompt_template.format(
    user_query=query, 
    context_lines=context_yaml, 
    idiom=idiom_name, 
    idiom_examples=idiom_examples
) 

# response = llm.invoke(prompt)
print(prompt)

In [ ]:
query1 = "how many wells are there?"
query2 = "What is the total water injection volume by year?"
query3 = "Tell me the mean yearly water injection volume for each subzone"
query4 = "rank wells by their variability (std) in water injection volume (the higher the grater the rank)?"
query5 = "whats the frequency of observations in the dataset (D, M, Y) ?"
query6 = "summarize the injection data"
query7 = "Which well had the single highest WATER_INJECTION_VOLUME reading at any point in time and what was that reading?"
query8 = "What is the average monthly injection volume per well grouped by NAME and MONTH?"
query9 = """For each SUBZONE compute the year-over-year percentage change in total injection 
volume and report the largest drop
"""


queries = [
    #(query1, lambda x: int(x.loc[0,:].values[0]) == 5, 1),
    #(query2, lambda x: abs(float(x.loc[ x["YEAR"] == 2016, :].values[0][1]) - 56202.305) < 0.01, 2),
    #(query3, lambda x: abs(1306.96 - float(x.set_index(["YEAR", "SUBZONE"]).iloc[:, 0].loc[(2019, "WARA1")])) < 0.01, 2),
    #(query4, lambda x: (x.loc[x["NAME"] == "I1", x.columns[-1]] == 1).any(), 2),
    #(query7, lambda x: (x.shape[0] == 1 and (x.iloc[0]["NAME"] == "I1") and abs(float(x.iloc[0]["WATER_INJECTION_VOLUME"]) - 3537.0) < 0.1), 1),
    (query8, lambda x: False, 2),
    #(query9, lambda x: False, 3),
]

for n, query_item in enumerate(queries):
    query, checking_fn, complexity = query_item
    print(60 * "=")
    print(f"Complexity index: {complexity}")
    print(query)
   
    prompt = prompt_template.format(
        user_query=query, 
        context_lines=context_yaml, 
        idiom=idiom_name, 
        idiom_examples=idiom_examples
    )
    response = llm.invoke(prompt)

    print(response)

    # execute the generated SQL
    sql = response.content.strip()
    print(sql)

    try:
        result = con.execute(sql).fetchdf()
        display(result.sample(min(3, result.shape[0])))

        # check result
        print("success", checking_fn(result))
    except Exception as e:
        print("error", e)


In [ ]:
from pprint import pprint 
pprint(response.usage_metadata)
pprint(response.response_metadata["token_usage"]) 

# Now lets make it agentic and add some sort of structured output 

## Generate a prompt_builder that RAGs the questions, reasoning and sql 
## Add custom tools 


In [ ]:
response.response_metadata['token_usage']['prompt_tokens']

In [ ]:


class SmartData:
    def __init__(self, llm=None):
        self.con = duckdb.connect()
        self.tables = set()
        self.semantic = {}
        self.semantic_model = load_semantic_model()
        self.columns = {}
        self._llm = llm

    # -----------------------------
    # LLM PROPERTY
    # -----------------------------
    @property
    def llm(self):
        return self._llm

    @llm.setter
    def llm(self, model):
        self._llm = model

    # -----------------------------
    # CORE EXECUTION
    # -----------------------------
    def execute_query(self, sql: str):
        return self.con.execute(sql).fetchdf()

    # -----------------------------
    # TABLE REGISTRATION
    # -----------------------------
    def register_tables(self, tables: Dict[str, Any]):
        register_tables(self.con, tables, self.tables, self.columns)

    def register_table(self, name: str, df):
        register_table(self.con, name, df, self.tables, self.columns)

    # -----------------------------
    # SEMANTIC REGISTRATION
    # -----------------------------
    def register_semantic(self, name: str, description: str):
        if name not in self.tables:
            raise ValueError(f"Table '{name}' is not registered")
        self.semantic[name] = description

    # -----------------------------
    # SQL GENERATION
    # -----------------------------
    def generate_sql(self, user_query: str, **kwargs) -> str:
        if self._llm is None:
            raise ValueError("LLM is not set")

        context_lines = []
        for table in self.tables:
            desc = self.semantic.get(table, "")
            cols = self.columns.get(table, [])
            context_lines.append(
                f"Table: {table}\nDescription: {desc}\nColumns: {', '.join(cols)}"
            )

        context = "\n\n".join(context_lines)

        prompt = f"""
You are an expert SQL generator for DuckDB.

Available tables:
{context}

User request:
{user_query}

Generate a valid DuckDB SQL query only.
"""

        return self._llm(prompt, **kwargs)


llm = azure_llm_if()
print( llm )

